# API Providers Test Bench

A quick connectivity check for multiple LLM API providers, one cell per provider. Each cell is self contained: it loads the key from `.env`, makes a single chat request, prints the reply plus basic usage info, and handles common failures (bad key, no credits, timeout, network).

## Setup

Add the keys you want to test to a `.env` file in this folder (do not commit it):

```
OPENROUTER_API_KEY=sk-or-v1-...
ANTHROPIC_API_KEY=sk-ant-...
OPENAI_API_KEY=sk-...
GEMINI_API_KEY=...
```

Packages (already added via `uv add`): `python-dotenv`, `requests`, `openai`, `anthropic`, `google-genai`.

Run the setup cell first, then run any provider cell independently.

In [1]:
# Setup: load environment variables and report which keys are present
import os
from dotenv import load_dotenv

load_dotenv(override=True)

PROMPT = "What is the meaning of life? Answer in one short sentence."

for name in ["OPENROUTER_API_KEY", "ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GEMINI_API_KEY"]:
    print(f"{name}: {'set' if os.getenv(name) else 'MISSING'}")

OPENROUTER_API_KEY: set
ANTHROPIC_API_KEY: set
OPENAI_API_KEY: set
GEMINI_API_KEY: MISSING


## 1. OpenRouter

Raw HTTP call (matches the provider's `fetch` example, translated to Python `requests`). OpenRouter is a router in front of many models, so change `model` to test different ones (e.g. `anthropic/claude-opus-4.1`, `google/gemini-2.5-pro`, `openai/gpt-4o`).

In [5]:
import os, requests
from dotenv import load_dotenv

load_dotenv(override=True)
PROMPT = "What is the meaning of life? Answer in one short sentence."

# Maps HTTP status codes to plain-language causes (shared by both OpenRouter cells).
HTTP_HINTS = {
    400: "bad request (check the model name / parameters)",
    401: "invalid or missing API key",
    402: "payment required: no credits on the account yet / billing not set up",
    403: "permission denied for this key or model",
    404: "unknown model or endpoint",
    408: "the server timed out waiting for the request",
    429: "rate limited or out of quota (add credits / slow down)",
}

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise RuntimeError("OPENROUTER_API_KEY not set in .env")

try:
    resp = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "HTTP-Referer": "http://localhost",   # optional: your site URL
            "X-Title": "API Providers Test",       # optional: your site name
            "Content-Type": "application/json",
        },
        json={
            "model": "openai/gpt-4o",
            "messages": [{"role": "user", "content": PROMPT}],
        },
        timeout=60,
    )
    resp.raise_for_status()
    data = resp.json()
    print("Model:", data.get("model"))
    print("Reply:", data["choices"][0]["message"]["content"])
    print("Usage:", data.get("usage"))
except requests.exceptions.Timeout:
    print("OpenRouter error: request timed out. The API did not respond in time.")
except requests.exceptions.ConnectionError:
    print("OpenRouter error: could not connect. Check your network / the service status.")
except requests.exceptions.HTTPError as e:
    code = e.response.status_code
    hint = HTTP_HINTS.get(code, "unexpected HTTP error")
    print(f"OpenRouter HTTP {code}: {hint}")
    print("Response body:", e.response.text[:500])
except Exception as e:
    print(f"OpenRouter unexpected error: {type(e).__name__}: {e}")

Model: openai/gpt-4o
Reply: The meaning of life is a subjective quest for purpose and fulfillment.
Usage: {'prompt_tokens': 20, 'completion_tokens': 14, 'total_tokens': 34, 'cost': 0.00019, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.00019, 'upstream_inference_prompt_cost': 5e-05, 'upstream_inference_completions_cost': 0.00014}, 'completion_tokens_details': {'reasoning_tokens': 0, 'image_tokens': 0, 'audio_tokens': 0}}


### 1b. OpenRouter, specific model with reasoning (multi-turn)

Tests a specific reasoning model (`z-ai/glm-5.3-flash`) and preserves `reasoning_details` across turns so the model continues its chain of thought on the follow up. To test another model, **copy this cell and change the `MODEL` line** (e.g. `deepseek/deepseek-r1`, `openai/o4-mini`).

In [2]:
import os, json, requests
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise RuntimeError("OPENROUTER_API_KEY not set in .env")

# Change MODEL and copy this cell to test other reasoning models on OpenRouter.
MODEL = "z-ai/glm-5.3-flash"

HTTP_HINTS = {
    400: "bad request (check the model name / parameters)",
    401: "invalid or missing API key",
    402: "payment required: no credits on the account yet / billing not set up",
    403: "permission denied for this key or model",
    404: "unknown model or endpoint",
    408: "the server timed out waiting for the request",
    429: "rate limited or out of quota (add credits / slow down)",
}

def openrouter_chat(messages, model=MODEL, reasoning=True):
    resp = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        data=json.dumps({
            "model": model,
            "messages": messages,
            "reasoning": {"enabled": reasoning},
        }),
        timeout=120,
    )
    resp.raise_for_status()
    return resp.json()

try:
    # First turn
    messages = [{"role": "user", "content": "How many r's are in the word 'strawberry'?"}]
    data = openrouter_chat(messages)
    assistant = data["choices"][0]["message"]
    print("Model:", data.get("model"))
    print("Reply 1:", assistant.get("content"))

    # Preserve the assistant message WITH reasoning_details, then ask a follow up
    messages.append({
        "role": "assistant",
        "content": assistant.get("content"),
        "reasoning_details": assistant.get("reasoning_details"),  # pass back unmodified
    })
    messages.append({"role": "user", "content": "Are you sure? Think carefully."})

    # Second turn: model continues reasoning from where it left off
    data2 = openrouter_chat(messages)
    print("Reply 2:", data2["choices"][0]["message"].get("content"))
    print("Usage:", data2.get("usage"))
except requests.exceptions.Timeout:
    print(f"OpenRouter error ({MODEL}): request timed out. The API did not respond in time.")
except requests.exceptions.ConnectionError:
    print(f"OpenRouter error ({MODEL}): could not connect. Check your network / the service status.")
except requests.exceptions.HTTPError as e:
    code = e.response.status_code
    hint = HTTP_HINTS.get(code, "unexpected HTTP error")
    print(f"OpenRouter HTTP {code} ({MODEL}): {hint}")
    print("Response body:", e.response.text[:500])
except Exception as e:
    print(f"OpenRouter unexpected error ({MODEL}): {type(e).__name__}: {e}")

Model: z-ai/glm-5.3-flash
Reply 1: There are **3** r's in the word 'strawberry':

**st r awbe rr y**

- 1 r in "straw"
- 2 r's in "berry"
Reply 2: Yes, I'm confident. Let me spell it out letter by letter:

s - t - **r** - a - w - b - e - **r** - **r** - y

That's **3 r's**: one after "st", and two consecutive r's in "berry".
Usage: {'prompt_tokens': 77, 'completion_tokens': 88, 'total_tokens': 165, 'cost': 2.7775e-05, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 2.7775e-05, 'upstream_inference_prompt_cost': 5.775e-06, 'upstream_inference_completions_cost': 2.2e-05}, 'completion_tokens_details': {'reasoning_tokens': 24, 'image_tokens': 0, 'audio_tokens': 0}}


## 2. Anthropic (Claude)

Uses the official `anthropic` SDK. Default model is Claude Opus 5 (`claude-opus-5`).

In [ ]:
import os
import anthropic
from dotenv import load_dotenv

load_dotenv(override=True)
PROMPT = "What is the meaning of life? Answer in one short sentence."

if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError("ANTHROPIC_API_KEY not set in .env")

client = anthropic.Anthropic(timeout=60)  # reads ANTHROPIC_API_KEY; 60s timeout

try:
    message = client.messages.create(
        model="claude-opus-5",
        max_tokens=1024,
        messages=[{"role": "user", "content": PROMPT}],
    )
    reply = next((b.text for b in message.content if b.type == "text"), "")
    print("Model:", message.model)
    print("Reply:", reply)
    print("Usage:", message.usage)
except anthropic.AuthenticationError:
    print("Anthropic error: invalid API key.")
except anthropic.PermissionDeniedError:
    print("Anthropic error: permission denied (key not enabled for this model).")
except anthropic.RateLimitError:
    print("Anthropic error: rate limited or out of quota.")
except anthropic.BadRequestError as e:
    # A low/zero credit balance surfaces here (e.g. 'credit balance is too low').
    print(f"Anthropic error: bad request, often unpaid / low credit balance. Detail: {e.message}")
except anthropic.APITimeoutError:
    print("Anthropic error: request timed out. The API did not respond in time.")
except anthropic.APIConnectionError:
    print("Anthropic error: could not connect. Check your network.")
except anthropic.APIStatusError as e:
    print(f"Anthropic API error {e.status_code}: {e.message}")
except Exception as e:
    print(f"Anthropic unexpected error: {type(e).__name__}: {e}")

## 3. OpenAI

Uses the official `openai` SDK (already used elsewhere in this repo).

In [3]:
import os
import openai
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)
PROMPT = "What is the meaning of life? Answer in one short sentence."

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not set in .env")

client = OpenAI(timeout=60)  # reads OPENAI_API_KEY; 60s timeout

try:
    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": PROMPT}],
    )
    print("Model:", resp.model)
    print("Reply:", resp.choices[0].message.content)
    print("Usage:", resp.usage)
except openai.AuthenticationError:
    print("OpenAI error: invalid API key.")
except openai.PermissionDeniedError:
    print("OpenAI error: permission denied for this key or model.")
except openai.RateLimitError:
    # 'insufficient_quota' arrives here: billing not set up or no credits yet.
    print("OpenAI error: rate limited or out of quota (billing not set up / no credits).")
except openai.APITimeoutError:
    print("OpenAI error: request timed out. The API did not respond in time.")
except openai.APIConnectionError:
    print("OpenAI error: could not connect. Check your network.")
except openai.APIStatusError as e:
    print(f"OpenAI API error {e.status_code}: {e.message}")
except Exception as e:
    print(f"OpenAI unexpected error: {type(e).__name__}: {e}")

OpenAI error: rate limited or out of quota (billing not set up / no credits).


## 4. Google Gemini

Uses the `google-genai` SDK. Get a key from Google AI Studio and set `GEMINI_API_KEY`.

In [4]:
import os
from google import genai
from google.genai import errors as genai_errors
from dotenv import load_dotenv

load_dotenv(override=True)
PROMPT = "What is the meaning of life? Answer in one short sentence."

if not os.getenv("GEMINI_API_KEY"):
    raise RuntimeError("GEMINI_API_KEY not set in .env")

# reads GEMINI_API_KEY; 60s (60000ms) timeout
client = genai.Client(http_options={"timeout": 60_000})
MODEL = "gemini-2.5-flash"

try:
    resp = client.models.generate_content(model=MODEL, contents=PROMPT)
    print("Model:", MODEL)
    print("Reply:", resp.text)
    print("Usage:", resp.usage_metadata)
except genai_errors.ClientError as e:
    # 4xx: 400 bad request, 401/403 bad key or permission, 429 quota / no billing.
    code = getattr(e, "code", None)
    if code == 429:
        print("Gemini error: rate limited or out of quota (free tier exhausted / billing not enabled).")
    elif code in (401, 403):
        print("Gemini error: invalid API key or permission denied.")
    else:
        print(f"Gemini client error {code}: {e}")
except genai_errors.ServerError as e:
    print(f"Gemini server error {getattr(e, 'code', None)}: the API is unavailable, try again later.")
except Exception as e:
    # Covers timeouts and connection failures raised by the underlying HTTP layer.
    print(f"Gemini error: {type(e).__name__}: {e}")

RuntimeError: GEMINI_API_KEY not set in .env